<a href="https://colab.research.google.com/github/RafaFBorges/IA004-Tarefa1/blob/main/IA004_Exerc%C3%ADcio_1_SVM_e_Holdout_Repetido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Instituto de Informática – UFRGS**
## Disciplina IA004 - Aprendizado de Máquina Supervisionado
#### *Profa. Mariana Recamonde Mendoza (mrmendoza@inf.ufrgs.br)*
<br>

---

***Observação:*** *Este notebook é disponibilizado como material complementar às aulas e slides fornecidos pela professora. Os conceitos fundamentais são apresentados no material teórico; aqui, o objetivo é reforçar a compreensão e demonstrar aspectos práticos do uso de algoritmos e estratégias de avaliação em Aprendizado de Máquina.*

---

# Notebook Prático — SVM com Holdout Repetido de Três Vias

**Objetivo:** Explorar o algoritmo **Support Vector Machine (SVM)** no contexto de um problema de predição de *churn*, utilizando o **holdout repetido de três vias** para:

1. Tratar pré-processamento (codificação, normalização e valores faltantes) de forma correta e sem vazamento de dados (*data leakage*)
2. Comparar diferentes funções de **kernel** do SVM (Linear, RBF, Polinomial) e analisar como elas afetam a complexidade do modelo
3. Investigar empiricamente os fenômenos de **overfitting**, **underfitting**, **viés** e **variância**
4. Comparar o SVM com a **Regressão Logística** como baseline

> **Dataset:** Telco Customer Churn — predição de saída de clientes de uma operadora de telecomunicações.  
> Fonte: [Kaggle — Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)


---
# 1. Carregamento de Bibliotecas

Antes de qualquer coisa, importamos todas as bibliotecas necessárias. É uma boa prática manter as importações no início do notebook.


In [ ]:
# ===================================================================
# Importações necessárias
# ===================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

# Semente global — garante reprodutibilidade
SEED = 42
np.random.seed(SEED)

print("Bibliotecas carregadas com sucesso!")


---
# 2. Carregamento e Descrição do Dataset

O dataset **Telco Customer Churn** contém informações sobre clientes de uma operadora de telecomunicações fictícia na Califórnia. O objetivo é prever se um cliente vai cancelar o contrato (*churn = 1*) ou permanecer (*churn = 0*).

As colunas incluem:
- **Dados demográficos:** gênero, se tem parceiro/dependentes, faixa etária
- **Serviços contratados:** telefone, internet, streaming, suporte técnico, etc.
- **Informações da conta:** tipo de contrato, forma de pagamento, cobrança mensal e total
- **Alvo:** `Churn` — se o cliente cancelou no último mês

Este é um problema clássico com **desbalanceamento de classes** (menos churns do que retenções), **variáveis categóricas** (que precisam de codificação) e **valores faltantes**. Neste notebook, não trataremos ainda a questão do desbalanceamento de classes.


In [ ]:
# Carregando o dataset
url = "https://drive.google.com/uc?export=view&id=10VrzI8mA2wPvkNIDaLzv-IglPU-Febtf"
df = pd.read_csv(url)

print(f"Formato do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()


---
# 3. Análise Exploratória dos Dados (EDA)

Antes de qualquer modelagem, precisamos entender o dataset: tipos de variáveis, distribuições, valores faltantes e balanceamento de classes.


In [ ]:
# Tipos de dados e valores faltantes
print("=== Tipos de dados ===")
print(df.dtypes)
print()
print("=== Valores faltantes (NaN diretos) ===")
print(df.isnull().sum())


> **Atenção:** A coluna `TotalCharges` é lida como texto (`object`) porque alguns registros contêm espaços em branco no lugar de valores numéricos. Isso é um exemplo de **valor faltante codificado de forma implícita** — precisaremos tratar isso no pré-processamento.


In [ ]:
# Corrigindo o tipo de TotalCharges e revelando os valores faltantes reais
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("=== Valores faltantes APÓS conversão de TotalCharges ===")
print(df.isnull().sum())
print(f"\nTotal de valores faltantes: {df.isnull().sum().sum()}")


> **Para pensar:** Os valores faltantes em `TotalCharges` pertencem a quais clientes? Faz sentido que esses valores estejam ausentes? Consulte as linhas correspondentes para verificar se existe algum padrão.

> ***Sua resposta aqui:***


In [ ]:
# Investigando os clientes com TotalCharges faltante
print("Clientes com TotalCharges ausente:")
df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head(10)


In [ ]:
# Distribuição da variável alvo (Churn)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem absoluta
churn_counts = df['Churn'].value_counts()
axes[0].bar(['Não Churn (No)', 'Churn (Yes)'], churn_counts.values,
            color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Distribuição do Alvo: Churn')
axes[0].set_ylabel('Frequência')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=11)

# Proporção
axes[1].pie(churn_counts.values, labels=['Não Churn', 'Churn'],
            colors=['steelblue', 'coral'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporção das Classes')

plt.tight_layout()
plt.show()
print(f"\nProporção: {churn_counts.values[1]/churn_counts.sum()*100:.1f}% de Churn (classe positiva)")


> **Para pensar:** O dataset está desbalanceado. Como isso pode afetar a interpretação da **acurácia**? Um modelo que sempre prevê "Não Churn" teria qual acurácia? Isso seria um bom modelo?

> ***Sua resposta aqui:***


In [ ]:
# Identificando colunas categóricas e numéricas
# Removemos CustomerID (não é feature) e Churn (é o alvo)
df_model = df.drop(['customerID'], axis=1)
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Colunas numéricas ({len(num_cols)}): {num_cols}")
print()
print(f"Colunas categóricas ({len(cat_cols)}): {cat_cols}")


In [ ]:
# Distribuição das variáveis numéricas
fig, axes = plt.subplots(1, len(num_cols), figsize=(14, 4))
for i, col in enumerate(num_cols):
    axes[i].hist(X[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor')
plt.suptitle('Distribuição das Variáveis Numéricas', y=1.02)
plt.tight_layout()
plt.show()


> **Para pensar:** Observando as distribuições dos atributos numéricos, por que a **normalização** é especialmente importante para o SVM?

> ***Sua resposta aqui:***


---
# 4. Pré-Processamento de Dados

O SVM é sensível à escala dos dados e não aceita variáveis categóricas diretamente. Precisamos realizar:

| Etapa | O que fazer | Por quê |
|---|---|---|
| **Valores faltantes** | Imputar com a média (numéricas) | SVM não aceita `NaN` |
| **Codificação** | One-Hot Encoding (categóricas) | SVM exige entradas numéricas |
| **Normalização** | StandardScaler (média 0, desvio 1) | SVM é baseado em distâncias — escalas diferentes distorcem as margens |

> **Regra de ouro:** todos os parâmetros de pré-processamento (média, desvio-padrão, categorias) devem ser **aprendidos apenas nos dados de treino** e aplicados nos demais conjuntos. Fazer isso com os dados completos antes da divisão é chamado de **data leakage** e contamina o processo de avaliação.

## 4.1 Separação do Conjunto de Teste (realizada UMA ÚNICA VEZ)

Reservamos 15% para os dados de teste. Esta divisão é aleatória e estratificada.

In [ ]:
# ─── SEPARAÇÃO INICIAL DO TESTE ───────────────────────────────
# 15% reservados para teste final — não serão utilizados em nenhuma
# decisão de modelagem (hiperparâmetros, seleção de modelo, etc.)
X_dev, X_teste, y_dev, y_teste = train_test_split(
    X, y,
    test_size=0.15,
    random_state=SEED,
    stratify=y   # mantém proporção de classes em todas as partições
)

print(f"Conjunto de desenvolvimento (treino+validação): {X_dev.shape[0]} instâncias")
print(f"Conjunto de teste (bloqueado até o final)      : {X_teste.shape[0]} instâncias")
print(f"Total                                          : {X_dev.shape[0] + X_teste.shape[0]} instâncias")
print()
print("Proporção de Churn no desenvolvimento:",
      round(y_dev.mean(), 3))
print("Proporção de Churn no teste           :",
      round(y_teste.mean(), 3))


## 4.2 Função de Pré-Processamento

Encapsulamos o pré-processamento em uma função para garantir que seja aplicado de forma idêntica a cada partição treino/validação do holdout repetido, sem risco de *data leakage*. Posteriormente iremos estudar a classe Pipeline, do scikit-lern, que ajuda nesse processo.


In [ ]:
def preprocessar(X_tr, X_val, num_cols, cat_cols):
    """
    Aplica imputação, codificação one-hot e normalização.
    - Todos os parâmetros são aprendidos EXCLUSIVAMENTE em X_tr.
    - X_val recebe apenas a transformação, sem re-ajuste.

    Etapas:
      1. Imputação numérica  → média  (trata NaN em TotalCharges, etc.)
      2. Imputação categórica → moda   (trata NaN em variáveis como gender)
      3. One-Hot Encoding das categóricas (já sem NaN)
      4. Normalização com StandardScaler

    Retorna X_tr transformado, X_val transformado e lista de nomes de colunas.
    """
    # ── 1. Imputação numérica: média ──────────────────────────────
    imputer_num = SimpleImputer(strategy='mean')
    X_tr_num = pd.DataFrame(
        imputer_num.fit_transform(X_tr[num_cols]),
        columns=num_cols,
        index=X_tr.index
    )
    X_val_num = pd.DataFrame(
        imputer_num.transform(X_val[num_cols]),
        columns=num_cols,
        index=X_val.index
    )

    # ── 2. Imputação categórica: moda ─────────────────────────────
    # Necessário antes do OHE, pois o encoder não aceita NaN.
    # A moda é aprendida no treino e aplicada na validação.
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X_tr_cat_imp = pd.DataFrame(
        imputer_cat.fit_transform(X_tr[cat_cols]),
        columns=cat_cols,
        index=X_tr.index
    )
    X_val_cat_imp = pd.DataFrame(
        imputer_cat.transform(X_val[cat_cols]),
        columns=cat_cols,
        index=X_val.index
    )

    # ── 3. Codificação One-Hot das variáveis categóricas ──────────
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    X_tr_cat = pd.DataFrame(
        encoder.fit_transform(X_tr_cat_imp),
        columns=encoder.get_feature_names_out(cat_cols),
        index=X_tr.index
    )
    X_val_cat = pd.DataFrame(
        encoder.transform(X_val_cat_imp),
        columns=encoder.get_feature_names_out(cat_cols),
        index=X_val.index
    )

    # ── 4. Concatenação ───────────────────────────────────────────
    X_tr_pp  = pd.concat([X_tr_num,  X_tr_cat],  axis=1)
    X_val_pp = pd.concat([X_val_num, X_val_cat], axis=1)

    # ── 5. Normalização (StandardScaler) ─────────────────────────
    scaler = StandardScaler()
    X_tr_scaled  = scaler.fit_transform(X_tr_pp)
    X_val_scaled = scaler.transform(X_val_pp)

    feature_names = X_tr_pp.columns.tolist()
    return X_tr_scaled, X_val_scaled, feature_names

print("Função de pré-processamento definida com sucesso!")


> **Para pensar:** Por que aplicamos `fit_transform` apenas no conjunto de treino e `transform` no conjunto de validação? O que aconteceria se fizéssemos `fit_transform` em ambos?

> ***Sua resposta aqui:***


---
# 5. Estratégia de Avaliação: Holdout Repetido de Três Vias

No **holdout de três vias**, o dataset é dividido em três conjuntos com papéis distintos:

| Conjunto | Papel | Tamanho neste experimento |
|---|---|---|
| **Treino** | O modelo aprende — `.fit()` | 70% |
| **Validação** | Ajuste de hiperparâmetros e seleção de modelos | 15% |
| **Teste** | Avaliação final — usado **uma única vez** | 15% (já separado) |

### Lembre-se: Por que repetir?

Uma única divisão aleatória pode gerar estimativas de desempenho otimistas ou pessimistas por acaso (alta variância). Ao **repetir o holdout com sementes diferentes**, obtemos uma **distribuição de desempenhos**, tornando a estimativa mais confiável e nos permitindo analisar a **variância** do modelo.


In [ ]:
# ─── CONFIGURAÇÕES DO EXPERIMENTO ─────────────────────────────
N_REPETICOES = 5          # número de repetições do holdout
# por uma questão de TEMPO, executamos apenas 5 vezes
# Proporção interna: 70% treino + 15% validação do total
# Como já removemos 15% de teste, o 'dev' tem 85% do total.
# Para obter 15% de validação do total: 15/85 ≈ 0.1765
TAMANHO_VAL = 0.15 / 0.85

print(f"N_REPETICOES : {N_REPETICOES}")
print(f"TAMANHO_VAL  : {TAMANHO_VAL:.4f} ({TAMANHO_VAL*100:.1f}% do conjunto de desenvolvimento)")
print(f"=> Treino  : ~{(1-TAMANHO_VAL)*0.85*100:.0f}% do total")
print(f"=> Validação: ~{TAMANHO_VAL*0.85*100:.0f}% do total")
print(f"=> Teste   : ~15% do total (já separado)")


---
# Parte 1 — Comparação de Kernels do SVM

## 6. Fundamentos: O Papel do Kernel

O kernel é o coração da versatilidade do SVM. Ele define a **transformação do espaço de features**, permitindo que o SVM encontre fronteiras não-lineares enquanto resolve internamente um problema linear (no espaço transformado).

| Kernel | Hiperparâmetros principais | Complexidade / Flexibilidade |
|---|---|---|
| **Linear** | C | Baixa — fronteira linear no espaço original |
| **RBF** (Radial Basis Function) | C, gamma | Alta — fronteira altamente não-linear |
| **Polinomial** | C, degree, coef0 | Média a alta — depende do grau |

> **Viés vs. Variância:** Kernels mais simples (linear) tendem a ter **alto viés** (underfitting). Kernels mais complexos (RBF com gamma alto) tendem a ter **alta variância** (overfitting). O parâmetro C controla a tolerância a erros de margem. C alto → modelo mais ajustado ao treino.

## 6.1 Experimento: SVM Linear, RBF e Polinomial com C padrão


In [ ]:
# ─── CONFIGURAÇÃO DOS MODELOS A COMPARAR ──────────────────────
modelos = {
    'SVM Linear (C=1)' : SVC(kernel='linear',  C=1,    random_state=SEED),
    'SVM RBF (C=1)'    : SVC(kernel='rbf',      C=1,    random_state=SEED),
    'SVM Poly (d=3)'   : SVC(kernel='poly',     C=1, degree=3, random_state=SEED),
    'Reg. Logística'   : LogisticRegression(C=1, max_iter=1000, random_state=SEED),
}

# ─── HOLDOUT REPETIDO ─────────────────────────────────────────
# Para cada repetição, usamos a MESMA partição treino/validação
# em todos os modelos — isso garante que diferenças de desempenho
# se devem ao modelo, não à aleatoriedade da divisão.
resultados = {nome: {'val_acc': [], 'val_prec': [], 'val_rec': [],
                     'tr_acc': []}
              for nome in modelos}

for rep in range(N_REPETICOES):
    semente = SEED + rep

    # 1. Divisão treino / validação
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_dev, y_dev,
        test_size=TAMANHO_VAL,
        random_state=semente,
        stratify=y_dev
    )

    # 2. Pré-processamento (sem data leakage)
    X_tr_sc, X_val_sc, _ = preprocessar(X_tr, X_val, num_cols, cat_cols)

    # 3. Treina e avalia cada modelo
    for nome, modelo in modelos.items():
        m = modelo.__class__(**modelo.get_params())   # cria nova instância
        m.fit(X_tr_sc, y_tr)

        # Desempenho na VALIDAÇÃO
        y_pred_val = m.predict(X_val_sc)
        resultados[nome]['val_acc'].append(accuracy_score(y_val, y_pred_val))
        resultados[nome]['val_prec'].append(precision_score(y_val, y_pred_val, zero_division=0))
        resultados[nome]['val_rec'].append(recall_score(y_val, y_pred_val, zero_division=0))

        # Desempenho no TREINO (para análise de overfitting)
        y_pred_tr = m.predict(X_tr_sc)
        resultados[nome]['tr_acc'].append(accuracy_score(y_tr, y_pred_tr))

print(f"Holdout repetido concluído: {N_REPETICOES} repetições x {len(modelos)} modelos")
print(f"Total de modelos treinados: {N_REPETICOES * len(modelos)}")


In [ ]:
# ─── TABELA DE RESULTADOS ─────────────────────────────────────
print(f"{'Modelo':<25} {'Acurácia Val.':>14} {'±DP':>8} {'Precisão Val.':>14} {'Recall Val.':>12}")
print("-" * 76)

for nome in modelos:
    r = resultados[nome]
    print(f"{nome:<25} {np.mean(r['val_acc']):>14.4f} {np.std(r['val_acc']):>8.4f} "          f"{np.mean(r['val_prec']):>14.4f} {np.mean(r['val_rec']):>12.4f}")


In [ ]:
# ─── VISUALIZAÇÃO: Distribuição do desempenho por modelo ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
nomes = list(modelos.keys())

for ax, metrica, titulo in zip(axes,
                                ['val_acc', 'val_prec', 'val_rec'],
                                ['Acurácia', 'Precisão', 'Recall']):
    dados = [resultados[n][metrica] for n in nomes]
    bp = ax.boxplot(dados, patch_artist=True,
                    medianprops=dict(color='coral', linewidth=2))
    colors = ['steelblue', 'mediumseagreen', 'mediumpurple', 'darkorange']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xticks(range(1, len(nomes)+1))
    ax.set_xticklabels([n.replace(' ', '\n') for n in nomes], fontsize=9)
    ax.set_title(f'{titulo} na Validação ({N_REPETICOES} reps.)')
    ax.set_ylabel(titulo)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparação de Modelos — Holdout Repetido', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


> **Para pensar:**
> 1. Algum dos kernels apresenta desempenho claramente superior nos três indicadores? O que isso sugere sobre a natureza (linear ou não-linear) das fronteiras de decisão neste problema?
> 2. Observe os desvios-padrão (tamanho das caixas no boxplot). Qual modelo apresenta maior **variabilidade** entre as repetições? O que isso indica sobre a **variância** desse modelo?
> 3. A Regressão Logística (nosso baseline linear) se sai tão bem quanto o SVM Linear?


> ***Sua resposta aqui:***


---
# Parte 2 — Overfitting, Underfitting, Viés e Variância

## 7. Analisando Treino vs. Validação: Detectando Overfitting

A diferença entre o desempenho no **treino** e na **validação** é o principal sinal diagnóstico de overfitting:

- **Overfitting:** desempenho no treino muito maior do que na validação → modelo memorizou o treino, não generalizou
- **Underfitting:** desempenho baixo tanto no treino quanto na validação → modelo é muito simples para capturar os padrões
- **Generalização adequada:** dezempenhos próximos entre treino e validação, ambas satisfatórias

## 7.1 Experimento: O Efeito do Parâmetro C no SVM RBF


In [ ]:
# ─── VARIANDO C NO SVM RBF ────────────────────────────────────
# C controla a tolerância a erros na margem:
# C pequeno → margem larga, mais erros permitidos → modelo mais simples (maior viés)
# C grande  → margem estreita, poucos erros → modelo mais complexo (maior variância)

C_valores = [0.01, 0.1, 1, 10, 100, 1000]

res_C = {c: {'tr_acc': [], 'val_acc': []} for c in C_valores}

for rep in range(N_REPETICOES):
    semente = SEED + rep
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_dev, y_dev,
        test_size=TAMANHO_VAL,
        random_state=semente,
        stratify=y_dev
    )
    X_tr_sc, X_val_sc, _ = preprocessar(X_tr, X_val, num_cols, cat_cols)

    for c in C_valores:
        clf = SVC(kernel='rbf', C=c, random_state=SEED)
        clf.fit(X_tr_sc, y_tr)
        res_C[c]['tr_acc'].append(accuracy_score(y_tr,  clf.predict(X_tr_sc)))
        res_C[c]['val_acc'].append(accuracy_score(y_val, clf.predict(X_val_sc)))

print("Experimento com C variável concluído!")


In [ ]:
# ─── VISUALIZAÇÃO: Curva Treino vs. Validação ────────────────
medias_tr  = [np.mean(res_C[c]['tr_acc'])  for c in C_valores]
medias_val = [np.mean(res_C[c]['val_acc']) for c in C_valores]
desvios_val= [np.std(res_C[c]['val_acc'])  for c in C_valores]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 — Curvas treino e validação
axes[0].semilogx(C_valores, medias_tr,  'o-', color='coral',     label='Treino',     linewidth=2)
axes[0].semilogx(C_valores, medias_val, 's--', color='steelblue', label='Validação',  linewidth=2)
axes[0].set_xlabel('C (escala log)')
axes[0].set_ylabel('Acurácia Média')
axes[0].set_title(f'SVM RBF: Treino vs. Validação por C ({N_REPETICOES} reps.)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xticks(C_valores)
axes[0].set_xticklabels([str(c) for c in C_valores])

# Anotações de interpretação
axes[0].annotate('Underfitting\n(alto viés)', xy=(C_valores[0], medias_val[0]),
                 xytext=(C_valores[0]*1.5, medias_val[0]-0.03),
                 fontsize=9, color='gray',
                 arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].annotate('Overfitting\n(alta variância)', xy=(C_valores[-1], medias_tr[-1]),
                 xytext=(C_valores[-2], medias_tr[-1]-0.04),
                 fontsize=9, color='coral',
                 arrowprops=dict(arrowstyle='->', color='coral'))

# Plot 2 — Gap treino-validação (medida direta de overfitting)
gap = [mt - mv for mt, mv in zip(medias_tr, medias_val)]
axes[1].bar(range(len(C_valores)), gap, color='coral', alpha=0.7, edgecolor='white')
axes[1].set_xticks(range(len(C_valores)))
axes[1].set_xticklabels([str(c) for c in C_valores])
axes[1].set_xlabel('C')
axes[1].set_ylabel('Gap (Treino − Validação)')
axes[1].set_title('Gap de Overfitting por valor de C')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


> **Interpretando os gráficos:**
> - **Região de underfitting (C muito pequeno):** a regularização forte demais impede o modelo de aprender os padrões dos dados. A acurácia no treino é baixa, e na validação também. Isso reflete **alto viés** — o modelo faz suposições muito simplistas.
> - **Região de overfitting (C muito grande):** a ausência de regularização permite que o modelo memorize o treino, incluindo o ruído. A acurácia no treino é alta, mas cai na validação. Isso reflete **alta variância** — o modelo é excessivamente sensível às particularidades da amostra de treino.
> - **Ponto ótimo:** a acurácia na validação atinge seu máximo. É o melhor equilíbrio viés-variância para este kernel e dataset.

> **Para pensar:**
> 1. Para qual valor de C o gap treino-validação é maior? Isso indica overfitting ou underfitting?
> 2. Para C=0.01, as acurácias de treino e validação são próximas. Isso necessariamente significa que o modelo está bem ajustado? Explique.
> 3. Se você tivesse que justificar a escolha de C para um colega sem conhecimento de ML, como você descreveria o papel desse parâmetro em termos práticos?

> ***Sua resposta aqui:***


## 7.2 Experimento: O Efeito do Parâmetro Gamma no SVM RBF

O parâmetro `gamma` define o **raio de influência** de cada ponto de suporte:
- `gamma` **pequeno** → influência ampla, fronteira mais suave → modelo mais simples (maior viés)
- `gamma` **grande** → influência restrita, fronteira muito irregular → modelo mais complexo (maior variância)


In [ ]:
# ─── VARIANDO GAMMA NO SVM RBF (C fixo) ──────────────────────
C_fixo = 1.0
gamma_valores = [0.001, 0.01, 0.1, 1, 10, 100]

res_gamma = {g: {'tr_acc': [], 'val_acc': []} for g in gamma_valores}

for rep in range(N_REPETICOES):
    semente = SEED + rep
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_dev, y_dev, test_size=TAMANHO_VAL,
        random_state=semente, stratify=y_dev
    )
    X_tr_sc, X_val_sc, _ = preprocessar(X_tr, X_val, num_cols, cat_cols)

    for g in gamma_valores:
        clf = SVC(kernel='rbf', C=C_fixo, gamma=g, random_state=SEED)
        clf.fit(X_tr_sc, y_tr)
        res_gamma[g]['tr_acc'].append(accuracy_score(y_tr,  clf.predict(X_tr_sc)))
        res_gamma[g]['val_acc'].append(accuracy_score(y_val, clf.predict(X_val_sc)))

# Visualização
medias_tr_g   = [np.mean(res_gamma[g]['tr_acc'])  for g in gamma_valores]
medias_val_g  = [np.mean(res_gamma[g]['val_acc']) for g in gamma_valores]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(gamma_valores, medias_tr_g,  'o-', color='coral',     label='Treino',    linewidth=2)
ax.semilogx(gamma_valores, medias_val_g, 's--', color='steelblue', label='Validação', linewidth=2)
ax.set_xlabel('gamma (escala log)')
ax.set_ylabel('Acurácia Média')
ax.set_title(f'SVM RBF (C={C_fixo}): Treino vs. Validação por gamma ({N_REPETICOES} reps.)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(gamma_valores)
ax.set_xticklabels([str(g) for g in gamma_valores])
plt.tight_layout()
plt.show()


> **Para pensar:**
> 1. Para gamma muito grande (ex: 100), o que acontece com a acurácia no treino? E na validação? Como você descreveria o comportamento do modelo?
> 2. Compare o gráfico de gamma com o de C. Os dois parâmetros têm efeitos similares sobre overfitting/underfitting? O que isso sugere sobre como eles controlam a complexidade do modelo?

> ***Sua resposta aqui:***


## 7.3 Experimento: Variância do Modelo — Estabilidade Entre Repetições

A **variância** de um modelo se manifesta como a **sensibilidade ao conjunto de treino usado**. No holdout repetido, podemos medir isso diretamente: quanto varia o desempenho do modelo conforme mudamos a divisão treino/validação?

Um modelo com **alta variância** produzirá resultados muito diferentes em cada repetição — seu desempenho depende fortemente de quais exemplos foram sorteados para o treino. É mais comum ter outliers na avaliação.

Atenção: esta análise seria beneficiada por um número maior de repetições, permitindo melhor aproximar a distribuição de desempenho.


In [ ]:
# ─── COMPARANDO A ESTABILIDADE DE MODELOS ────────────────────
# Usamos os resultados já calculados na Seção 6
# Comparamos um modelo simples (RL), um modelo moderado (SVM RBF C=1)
# e um modelo instável (SVM RBF C=1000)

# Rodamos o SVM RBF C=1000 novamente para ter os dados
res_rbf_1000 = {'tr_acc': [], 'val_acc': []}
for rep in range(N_REPETICOES):
    semente = SEED + rep
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_dev, y_dev, test_size=TAMANHO_VAL,
        random_state=semente, stratify=y_dev
    )
    X_tr_sc, X_val_sc, _ = preprocessar(X_tr, X_val, num_cols, cat_cols)
    clf = SVC(kernel='rbf', C=1000, random_state=SEED)
    clf.fit(X_tr_sc, y_tr)
    res_rbf_1000['tr_acc'].append(accuracy_score(y_tr,  clf.predict(X_tr_sc)))
    res_rbf_1000['val_acc'].append(accuracy_score(y_val, clf.predict(X_val_sc)))

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparacao = {
    'Reg. Logística\n(baseline)': resultados['Reg. Logística']['val_acc'],
    'SVM RBF\n(C=1)': resultados['SVM RBF (C=1)']['val_acc'],
    'SVM RBF\n(C=1000)': res_rbf_1000['val_acc'],
}

# Plot 1 — Boxplot das distribuições
bp = axes[0].boxplot(list(comparacao.values()), patch_artist=True,
                     medianprops=dict(color='coral', linewidth=2))
colors_box = ['darkorange', 'steelblue', 'firebrick']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_xticks(range(1, 4))
axes[0].set_xticklabels(list(comparacao.keys()))
axes[0].set_ylabel('Acurácia na Validação')
axes[0].set_title(f'Distribuição do Desempenho ({N_REPETICOES} repetições)')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2 — Acurácia por repetição (linha do tempo)
for (nome, vals), cor in zip(comparacao.items(), colors_box):
    axes[1].plot(range(1, N_REPETICOES+1), vals,
                 'o-', label=nome.replace('\n', ' '), alpha=0.7, color=cor, linewidth=1.5)
axes[1].set_xlabel('Repetição')
axes[1].set_ylabel('Acurácia na Validação')
axes[1].set_title('Acurácia por Repetição — Estabilidade do Modelo')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Viés vs. Variância: Comparação de Modelos', fontsize=13)
plt.tight_layout()
plt.show()

# Tabela de estatísticas
print(f"\n{'Modelo':<25} {'Média':>10} {'Desvio':>10} {'Min':>8} {'Max':>8}")
print("-" * 65)
for nome_exib, vals in comparacao.items():
    nm = nome_exib.replace('\n', ' ')
    print(f"{nm:<25} {np.mean(vals):>10.4f} {np.std(vals):>10.4f} "          f"{np.min(vals):>8.4f} {np.max(vals):>8.4f}")


> **Interpretação prática:**
> - **SVM RBF C=1000:** tende a ter acurácia de treino próxima de 1.0 (memoriza os dados) mas apresenta **maior desvio-padrão** na validação. O desempenho "pula" bastante entre repetições porque o modelo é muito sensível a quais exemplos específicos caíram no treino — isso é **alta variância**.
> - **Regressão Logística e SVM RBF C=1:** modelos mais regularizados tendem a ser **mais estáveis** (menor variância), mesmo que percam um pouco de acurácia em algumas repetições.

> **Para pensar:**
> 1. Qual dos três modelos você preferiria usar em produção, considerando apenas a análise de viés-variância? Por quê?
> 2. Um modelo com desvio-padrão de 0.001 e acurácia média de 0.72 pode ser preferível a um modelo com desvio-padrão de 0.015 e acurácia média de 0.78? Em quais situações práticas isso seria verdade?

> ***Sua resposta aqui:***


---
# Parte 3 — Seleção de Hiperparâmetros e Comparação Final

## 8. Otimizando Hiperparâmetros via Holdout Repetido

Agora que entendemos o papel de C e gamma, vamos usá-los de forma sistemática para encontrar a configuração de SVM RBF que maximiza o desempenho na validação.

> **Regra fundamental:** os hiperparâmetros são escolhidos com base na validação. O teste **nunca é consultado** durante esse processo.


In [ ]:
# ─── GRADE DE HIPERPARÂMETROS SVM RBF ─────────────────────────
C_grid     = [0.1, 1, 10, 100]
gamma_grid = [0.001, 0.01, 0.1, 1]

# resultados_grid[(C, gamma)] = lista de acurácias na validação
resultados_grid = {}

for c in C_grid:
    for g in gamma_grid:
        chave = (c, g)
        resultados_grid[chave] = []

        for rep in range(N_REPETICOES):
            semente = SEED + rep
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_dev, y_dev, test_size=TAMANHO_VAL,
                random_state=semente, stratify=y_dev
            )
            X_tr_sc, X_val_sc, _ = preprocessar(X_tr, X_val, num_cols, cat_cols)

            clf = SVC(kernel='rbf', C=c, gamma=g, random_state=SEED)
            clf.fit(X_tr_sc, y_tr)
            resultados_grid[chave].append(
                accuracy_score(y_val, clf.predict(X_val_sc))
            )

print(f"Grid search concluído: {len(C_grid) * len(gamma_grid)} configurações x {N_REPETICOES} repetições")
print(f"Total de modelos treinados: {len(C_grid) * len(gamma_grid) * N_REPETICOES}")


In [ ]:
# ─── HEATMAP DA ACURÁCIA MÉDIA NA VALIDAÇÃO ──────────────────
medias_grid = np.zeros((len(C_grid), len(gamma_grid)))
for i, c in enumerate(C_grid):
    for j, g in enumerate(gamma_grid):
        medias_grid[i, j] = np.mean(resultados_grid[(c, g)])

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(medias_grid, annot=True, fmt='.4f', cmap='YlGnBu',
            xticklabels=[str(g) for g in gamma_grid],
            yticklabels=[str(c) for c in C_grid],
            ax=ax)
ax.set_xlabel('gamma')
ax.set_ylabel('C')
ax.set_title(f'Acurácia Média na Validação — SVM RBF ({N_REPETICOES} reps.)')
plt.tight_layout()
plt.show()

# Melhor configuração
melhor_chave = max(resultados_grid, key=lambda k: np.mean(resultados_grid[k]))
C_otimo, gamma_otimo = melhor_chave
print(f"\nMelhor configuração: C={C_otimo}, gamma={gamma_otimo}")
print(f"Acurácia média na validação: {np.mean(resultados_grid[melhor_chave]):.4f} "      f"(± {np.std(resultados_grid[melhor_chave]):.4f})")


> **Para pensar:**
> 1. O heatmap mostra alguma região de overfitting? Como você a identificaria apenas olhando para os valores de C e gamma?
> 2. Se dois pares (C, gamma) apresentam acurácia média muito similar mas desvios-padrão diferentes, qual você escolheria e por quê?
> 3. Você poderia usar o conjunto de teste para confirmar que a configuração ótima encontrada é realmente a melhor? Pense na justificativa da resposta.

> ***Sua resposta aqui:***


## 9. Comparação Final: SVM Otimizado vs. Baseline (Regressão Logística)

Agora comparamos o **melhor SVM RBF encontrado** contra a **Regressão Logística** (nosso baseline), usando as mesmas partições do holdout repetido.


In [ ]:
# ─── COMPARAÇÃO FINAL NA VALIDAÇÃO ───────────────────────────
res_final_svm = {'val_acc': [], 'val_prec': [], 'val_rec': [], 'tr_acc': []}
res_final_lr  = {'val_acc': [], 'val_prec': [], 'val_rec': [], 'tr_acc': []}

for rep in range(N_REPETICOES):
    semente = SEED + rep
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_dev, y_dev, test_size=TAMANHO_VAL,
        random_state=semente, stratify=y_dev
    )
    X_tr_sc, X_val_sc, _ = preprocessar(X_tr, X_val, num_cols, cat_cols)

    # SVM RBF ótimo
    svm_opt = SVC(kernel='rbf', C=C_otimo, gamma=gamma_otimo, random_state=SEED)
    svm_opt.fit(X_tr_sc, y_tr)
    yv = svm_opt.predict(X_val_sc)
    res_final_svm['val_acc'].append(accuracy_score(y_val, yv))
    res_final_svm['val_prec'].append(precision_score(y_val, yv, zero_division=0))
    res_final_svm['val_rec'].append(recall_score(y_val, yv, zero_division=0))
    res_final_svm['tr_acc'].append(accuracy_score(y_tr, svm_opt.predict(X_tr_sc)))

    # Regressão Logística
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    lr.fit(X_tr_sc, y_tr)
    yv_lr = lr.predict(X_val_sc)
    res_final_lr['val_acc'].append(accuracy_score(y_val, yv_lr))
    res_final_lr['val_prec'].append(precision_score(y_val, yv_lr, zero_division=0))
    res_final_lr['val_rec'].append(recall_score(y_val, yv_lr, zero_division=0))
    res_final_lr['tr_acc'].append(accuracy_score(y_tr, lr.predict(X_tr_sc)))

# Tabela
print("=== COMPARAÇÃO FINAL NA VALIDAÇÃO ===\n")
print(f"{'Modelo':<30} {'Acurácia':>10} {'±DP':>8} {'Precisão':>10} {'Recall':>10}")
print("-" * 73)
for nome, res in [(f'SVM RBF (C={C_otimo}, γ={gamma_otimo})', res_final_svm),
                  ('Regressão Logística (baseline)', res_final_lr)]:
    print(f"{nome:<30} {np.mean(res['val_acc']):>10.4f} {np.std(res['val_acc']):>8.4f} "          f"{np.mean(res['val_prec']):>10.4f} {np.mean(res['val_rec']):>10.4f}")


In [ ]:
# ─── VISUALIZAÇÃO: Diferença por repetição ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot comparativo
bp = axes[0].boxplot([res_final_svm['val_acc'], res_final_lr['val_acc']],
                     patch_artist=True,
                     medianprops=dict(color='coral', linewidth=2),
                     tick_labels=[f'SVM RBF\nC={C_otimo}, γ={gamma_otimo}',
                                  'Reg.\nLogística'])
for patch, color in zip(bp['boxes'], ['steelblue', 'darkorange']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_ylabel('Acurácia na Validação')
axes[0].set_title(f'Distribuição — {N_REPETICOES} Repetições')
axes[0].grid(axis='y', alpha=0.3)

# Diferença por repetição
diferencas = np.array(res_final_svm['val_acc']) - np.array(res_final_lr['val_acc'])
axes[1].plot(range(1, N_REPETICOES+1), diferencas, 'o-', color='steelblue', alpha=0.7)
axes[1].axhline(0, color='coral', linestyle='--', linewidth=1.5, label='Diferença zero')
axes[1].axhline(np.mean(diferencas), color='navy', linestyle='-', linewidth=1.5,
                label=f'Diferença média: {np.mean(diferencas):+.4f}')
axes[1].set_xlabel('Repetição')
axes[1].set_ylabel('Acurácia SVM − Acurácia RL')
axes[1].set_title('Diferença de Desempenho por Repetição')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('SVM RBF Otimizado vs. Regressão Logística', fontsize=13)
plt.tight_layout()
plt.show()

n_svm_vence = sum(1 for d in diferencas if d > 0)
print(f"SVM RBF venceu em {n_svm_vence}/{N_REPETICOES} repetições.")
print(f"RL venceu em {N_REPETICOES - n_svm_vence}/{N_REPETICOES} repetições.")


> **Para pensar:**
> 1. O SVM otimizado é consistentemente melhor do que a Regressão Logística em todas as repetições, ou existem repetições em que a RL vence? O que isso indica?
> 2. Se a diferença média SVM − RL for pequena (ex: < 0.01), isso justifica a complexidade computacional adicional do SVM? Quais outros fatores você consideraria nessa decisão?
> 3. O resultado do holdout repetido é mais confiável do que uma única divisão? Por quê?

> ***Sua resposta aqui:***


---
# Parte 4 — Avaliação Final no Conjunto de Teste

## 10. Seleção do Modelo Final e Avaliação no Teste

Com base nos resultados de validação, selecionamos o modelo com **maior acurácia média**. Esta é a **única vez que o conjunto de teste é utilizado**.

> Treinamos o modelo final em **todo o conjunto de desenvolvimento** (`X_dev`), pois assim aproveitamos o máximo de dados disponíveis para o treino final. O teste serve apenas para estimar o desempenho em dados genuinamente novos.


In [ ]:
# ─── SELEÇÃO DO MODELO FINAL ──────────────────────────────────
media_svm = np.mean(res_final_svm['val_acc'])
media_lr  = np.mean(res_final_lr['val_acc'])

print(f"Acurácia média na validação:")
print(f"  SVM RBF (C={C_otimo}, γ={gamma_otimo}): {media_svm:.4f}")
print(f"  Regressão Logística           : {media_lr:.4f}")
print()

if media_svm >= media_lr:
    modelo_escolhido = 'SVM RBF'
    print(f"Modelo selecionado: SVM RBF (C={C_otimo}, γ={gamma_otimo})")
else:
    modelo_escolhido = 'Regressão Logística'
    print("Modelo selecionado: Regressão Logística")

print()
print("O teste será utilizado APENAS na próxima célula.")


In [ ]:
# ─── PRÉ-PROCESSAMENTO FINAL (treinado em X_dev completo) ────
X_dev_sc, X_teste_sc, feature_names = preprocessar(X_dev, X_teste, num_cols, cat_cols)

# ─── TREINAMENTO FINAL ────────────────────────────────────────
if modelo_escolhido == 'SVM RBF':
    modelo_final = SVC(kernel='rbf', C=C_otimo, gamma=gamma_otimo, random_state=SEED)
else:
    modelo_final = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)

modelo_final.fit(X_dev_sc, y_dev)

# ─── AVALIAÇÃO NO TESTE (UMA ÚNICA VEZ) ──────────────────────
y_pred_teste = modelo_final.predict(X_teste_sc)

acc_teste   = accuracy_score(y_teste,  y_pred_teste)
prec_teste  = precision_score(y_teste, y_pred_teste, zero_division=0)
rec_teste   = recall_score(y_teste,   y_pred_teste, zero_division=0)

print("=== RESULTADO FINAL NO TESTE ===\n")
print(f"Modelo selecionado: {modelo_escolhido}")
print(f"Acurácia  no teste : {acc_teste:.4f}")
print(f"Precisão  no teste : {prec_teste:.4f}")
print(f"Recall    no teste : {rec_teste:.4f}")


In [ ]:
# ─── MATRIZ DE CONFUSÃO FINAL ────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_teste, y_pred_teste,
    display_labels=['Não Churn', 'Churn'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Matriz de Confusão — {modelo_escolhido}\n(Conjunto de Teste)')
plt.tight_layout()
plt.show()


> **Para pensar:**
> 1. A acurácia no teste é próxima da acurácia média na validação? Uma grande diferença entre essas estimativas pode ser problemática?
> 2. Olhando a matriz de confusão: o modelo erra mais classificando churns como não-churn (falsos negativos) ou não-churns como churn (falsos positivos)? Para uma empresa de telecomunicações, qual tipo de erro é mais custoso?
> 3. Se a acurácia no teste fosse muito **maior** do que na validação, isso seria uma boa notícia ou levantaria suspeitas? Por quê?

> ***Sua resposta aqui:***


---
# Parte 5 — Experimento Avançado: Curva de Aprendizado

## 11. Diagnóstico de Viés e Variância pela Curva de Aprendizado

A **curva de aprendizado** mostra como o desempenho evolui conforme o tamanho do conjunto de treino aumenta. É uma ferramenta poderosa para diagnosticar viés e variância:

- **Modelo com alto viés (underfitting):** ambas as curvas (treino e validação) convergem para um valor baixo. Mais dados não ajudam muito.
- **Modelo com alta variância (overfitting):** há uma grande lacuna entre treino e validação. Mais dados tendem a ajudar, pois forçam o modelo a generalizar.

Vamos comparar a curva de aprendizado do SVM Linear e do SVM RBF com C alto.


In [ ]:
# ─── CURVA DE APRENDIZADO ─────────────────────────────────────
# Usamos uma única divisão treino/validação para esta análise
X_tr_cl, X_val_cl, y_tr_cl, y_val_cl = train_test_split(
    X_dev, y_dev, test_size=TAMANHO_VAL, random_state=SEED, stratify=y_dev
)
X_tr_cl_sc, X_val_cl_sc, _ = preprocessar(X_tr_cl, X_val_cl, num_cols, cat_cols)

frações = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
n_total = len(X_tr_cl_sc)

resultados_curva = {
    'SVM Linear (C=1)'   : {'tr': [], 'val': []},
    'SVM RBF (C=100)'    : {'tr': [], 'val': []},
}

for frac in frações:
    n_usar = max(10, int(frac * n_total))
    idx = np.random.RandomState(SEED).permutation(n_total)[:n_usar]
    X_sub = X_tr_cl_sc[idx]
    y_sub = y_tr_cl.iloc[idx] if hasattr(y_tr_cl, 'iloc') else y_tr_cl[idx]

    for nome, (kernel, C_val) in [('SVM Linear (C=1)', ('linear', 1)),
                                   ('SVM RBF (C=100)',  ('rbf',    100))]:
        clf = SVC(kernel=kernel, C=C_val, random_state=SEED)
        clf.fit(X_sub, y_sub)
        resultados_curva[nome]['tr'].append(accuracy_score(y_sub, clf.predict(X_sub)))
        resultados_curva[nome]['val'].append(accuracy_score(y_val_cl, clf.predict(X_val_cl_sc)))

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_valores = [int(f * n_total) for f in frações]

for ax, nome in zip(axes, resultados_curva):
    ax.plot(n_valores, resultados_curva[nome]['tr'],  'o-', color='coral',
            label='Treino', linewidth=2)
    ax.plot(n_valores, resultados_curva[nome]['val'], 's--', color='steelblue',
            label='Validação', linewidth=2)
    ax.fill_between(n_valores,
                    resultados_curva[nome]['tr'],
                    resultados_curva[nome]['val'],
                    alpha=0.1, color='gray', label='Gap (overfitting)')
    ax.set_xlabel('Tamanho do conjunto de treino')
    ax.set_ylabel('Acurácia')
    ax.set_title(f'Curva de Aprendizado — {nome}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Curvas de Aprendizado: Diagnóstico de Viés e Variância', fontsize=13)
plt.tight_layout()
plt.show()


> **Interpretando as curvas:**
> - **SVM Linear:** as duas curvas tendem a se aproximar rapidamente. Se ambas estabilizarem em um valor relativamente baixo, isso é sinal de **alto viés** — adicionar mais dados não resolve o problema; seria necessário aumentar a complexidade do modelo.
> - **SVM RBF C=100:** há tipicamente uma grande lacuna entre treino e validação com poucos dados. Com mais dados, a lacuna tende a diminuir, pois o modelo é "forçado" a aprender padrões mais gerais. Isso é o diagnóstico clássico de **alta variância**.

> **Para pensar:**
> 1. Para o SVM Linear, dobrar o tamanho do conjunto de treino melhoraria significativamente o desempenho? Por quê?
> 2. Se o SVM RBF C=100 ainda tiver um grande gap treino-validação mesmo com todos os dados de treino disponíveis, qual seria a solução mais indicada: coletar mais dados ou reduzir C?
> 3. Existe um modelo que nunca sofre de overfitting, independentemente de C? Explique.

> ***Sua resposta aqui:***


---
# 12. Resumo e Conclusões

Este notebook explorou o SVM no contexto de predição de churn, cobrindo:

## Conceitos demonstrados empiricamente

| Conceito | Como foi demonstrado | Takeaway principal |
|---|---|---|
| **Função de kernel** | Comparação linear, RBF, polinomial | O kernel deve ser escolhido com base na natureza do problema — kernels mais complexos não são sempre melhores |
| **Overfitting** | C alto: gap treino > validação | Acurácia alta no treino não garante boa generalização |
| **Underfitting** | C muito baixo: ambas as acurácias baixas | Modelo excessivamente regularizado não captura os padrões |
| **Viés** | SVM Linear vs. RBF no mesmo problema | Alto viés → erro sistemático independente dos dados |
| **Variância** | Desvio-padrão nas repetições do holdout | Alta variância → instabilidade conforme muda o conjunto de treino |
| **Curva de aprendizado** | Treino vs. validação por tamanho do treino | Diagnóstica qual problema dominante: viés ou variância |
| **Holdout repetido** | 20 repetições com sementes diferentes | Produz estimativas mais confiáveis do que uma única divisão |
| **Pré-processamento seguro** | Fit apenas no treino, transform nos demais | Previne *data leakage* e estimativas artificialmente otimistas |

## Questões finais de reflexão

> 1. Qual foi, ao final, o modelo com melhor desempenho no teste? A diferença em relação ao baseline (Regressão Logística) foi expressiva?
> 2. Para o problema de churn, a **precisão** e o **recall** têm importâncias diferentes dependendo do objetivo de negócio. Descreva um cenário em que maximizar o recall de churn seria a prioridade, e outro em que a precisão seria mais importante.
> 3. Se você fosse apresentar os resultados deste estudo para um gestor de negócios (sem conhecimento técnico), quais seriam as 3 principais mensagens que você comunicaria?

> ***Sua resposta aqui:***
